# 03 PDF CV Extraction

This notebook tests text extraction from a real CV in PDF format.

Main goals:
- load a PDF CV from the local project folder
- extract text using `pypdf`
- extract text using `pdfplumber`
- compare extraction quality
- clean extracted CV text
- save the final extracted text for later CV quality analysis and structured extraction

## 1. Imports and Settings

In [1]:
import re
import pandas as pd

from pathlib import Path
from pypdf import PdfReader
import pdfplumber

In [2]:
RAW_TEST_CV_DIR = Path("../data/raw/test_cv")
PROCESSED_TEST_CV_DIR = Path("../data/processed/test_cv")

#test_cv="Example_Anonymized_CV.pdf"
test_cv="CV_searchable_ATS.pdf"

In [3]:
cv_path=RAW_TEST_CV_DIR/test_cv
print(cv_path)

..\data\raw\test_cv\CV_searchable_ATS.pdf


## 2. Extract Text Using pypdf

`pypdf` is a simple PDF text extraction library.  
It works well for many standard PDFs, but may struggle with complex layouts, tables or multi-column CVs.

In [4]:
def extract_text_with_pypdf(pdf_path):
    
    reader = PdfReader(pdf_path)
    extracted_pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text()


        if page_text is None:
            page_text = ""

        extracted_pages.append(page_text)

    full_text = "\n\n".join(extracted_pages)

    return full_text

In [5]:
pypdf_text=extract_text_with_pypdf(cv_path)

In [6]:
print(pypdf_text)

ALEX JOHNSON ¢ +1 202-555-0147 | alex.jjohnson@example.com | Example City, 10000 
SOFTWARE ENGINEERING STUDENT 
in| https://www.linkedin.com/in/alex-johnson-example/ 
PROFILE 
Highly motivated Third-Year Software Engineering student (GPA 9.2/10.0) at the School of Electrical 
Engineering (ETF), University of Belgrade, specializing in Data Structures and Algorithms (DSA) and 
System Programming (C/C++, Java,Python). Seeking to leverage advanced technical skills and a robust 
background in low-level systems and application development to contribute high-impact solutions to a 
challenging and innovative engineering environment. Strong communication and teamwork skills 
developed through university projects. 
EDUCATION 
University of Belgrade - School of Electrical Engineering (ETF) 
e Degree: Bachelor of Science (B.Sc.) in Software Engineering 
Location: Example City 
Expected Graduation: June 2027 
Year of Study: Third Year Student 
Cumulative GPA: 9.2 / 10.0 
Relevant Coursework Highlig

## 3. Extract Text Using pdfplumber

`pdfplumber` is often better for CVs because it can preserve layout and line structure more accurately than basic PDF extractors.

In [7]:
def extract_text_with_pdfplumber(pdf_path):

    extracted_pages = []


    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            page_text = page.extract_text()

            if page_text is None:
                page_text = ""

            extracted_pages.append(page_text)

    full_text = "\n\n".join(extracted_pages)

    return full_text

In [8]:
pdfplumber_text=extract_text_with_pdfplumber(cv_path)

In [9]:
print(pdfplumber_text)

ALEX JOHNSON
¢ +1 202-555-0147 | alex.jjohnson@example.com | Example City, 10000
SOFTWARE ENGINEERING STUDENT
in| https://www.linkedin.com/in/alex-johnson-example/
PROFILE
Highly motivated Third-Year Software Engineering student (GPA 9.2/10.0) at the School of Electrical
Engineering (ETF), University of Belgrade, specializing in Data Structures and Algorithms (DSA) and
System Programming (C/C++, Java,Python). Seeking to leverage advanced technical skills and a robust
background in low-level systems and application development to contribute high-impact solutions to a
challenging and innovative engineering environment. Strong communication and teamwork skills
developed through university projects.
EDUCATION
University of Belgrade - School of Electrical Engineering (ETF)
e Degree: Bachelor of Science (B.Sc.) in Software Engineering
Location: Example City
Expected Graduation: June 2027
Year of Study: Third Year Student
Cumulative GPA: 9.2 / 10.0
Relevant Coursework Highlights:
e Data Struc

## 4. Compare Extraction Results

Both extraction methods are compared using basic text statistics.

The final method should be selected based on:
- text length
- readability
- preservation of sections
- whether skills, education and experience are visible

In [10]:
comparison = pd.DataFrame({
    "method": ["pypdf", "pdfplumber"],
    "character_count": [len(pypdf_text), len(pdfplumber_text)],
    "word_count": [len(pypdf_text.split()), len(pdfplumber_text.split())],
    "line_count": [len(pypdf_text.splitlines()), len(pdfplumber_text.splitlines())]
})



In [11]:
comparison

,method,character_count,word_count,line_count
0,pypdf,2487,327,44
1,pdfplumber,2443,327,45


Although `pypdf` extracted a higher number of characters and words, the `pdfplumber` output was selected as the final extraction method because it produced a more readable and better structured CV text. Since all relevant CV information was preserved, readability and structure were prioritized over raw text length.

In [12]:
 raw_cv_text = pdfplumber_text

## 5. Clean Extracted CV Text

In [13]:
def clean_extracted_cv_text(text):
 
    if text is None:
        return ""

    text = str(text)

    text = text.replace("\x00", " ")

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:

        line = re.sub(r"[ \t]+", " ", line)

        line = line.strip()

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    text = re.sub(r"\n{3,}", "\n\n", text)

    text = text.strip()

    return text

In [14]:
clean_cv_text = clean_extracted_cv_text(raw_cv_text)


In [15]:

print(clean_cv_text)

ALEX JOHNSON
¢ +1 202-555-0147 | alex.jjohnson@example.com | Example City, 10000
SOFTWARE ENGINEERING STUDENT
in| https://www.linkedin.com/in/alex-johnson-example/
PROFILE
Highly motivated Third-Year Software Engineering student (GPA 9.2/10.0) at the School of Electrical
Engineering (ETF), University of Belgrade, specializing in Data Structures and Algorithms (DSA) and
System Programming (C/C++, Java,Python). Seeking to leverage advanced technical skills and a robust
background in low-level systems and application development to contribute high-impact solutions to a
challenging and innovative engineering environment. Strong communication and teamwork skills
developed through university projects.
EDUCATION
University of Belgrade - School of Electrical Engineering (ETF)
e Degree: Bachelor of Science (B.Sc.) in Software Engineering
Location: Example City
Expected Graduation: June 2027
Year of Study: Third Year Student
Cumulative GPA: 9.2 / 10.0
Relevant Coursework Highlights:
e Data Struc

## 6. Save Extracted CV Text

In [16]:
output_txt_path = PROCESSED_TEST_CV_DIR / "test_cv_extracted_text.txt"

with open(output_txt_path, "w", encoding="utf-8") as file:
    file.write(clean_cv_text)

print(f"Extracted CV text saved to: {output_txt_path}")

Extracted CV text saved to: ..\data\processed\test_cv\test_cv_extracted_text.txt
